# Heterogeneous Residual Specialist Mixer (HeRSMix)

This notebook implements a **new, evidence-driven direction** for your project after these failures:

- PBT architecture search failed
- DBLoss full run failed
- adaptive lookback full run failed
- a simple gated residual wavelet pilot failed

## Core idea

Keep a **strong simple base predictor always on**, and learn **heterogeneous residual specialists** that only correct the base when the regime calls for them:

- **Base expert**: DLinear-style target-channel predictor
- **Periodic residual expert**: frequency-aware residual corrector
- **Local residual expert**: short-range Conv1D residual corrector
- **Wavelet residual expert**: your existing `AdaptiveWaveletMixer`
- **Router with abstention**: learns whether none / periodic / local / wavelet should contribute
- **Selective training**: downweight highly uncertain samples using expert-disagreement, inspired by selective learning

## Why this is more defensible

Your benchmark already shows **different winners on different datasets**, which matches recent evidence that there are **no universal champions** in LTSF. Recent work also argues that **heterogeneous experts** are more appropriate than homogeneous processing for varied time-series structure, and that **uniformly optimizing all samples/timesteps** can overfit noise. This notebook turns those lessons into one runnable pilot/full experiment.

## Strongest defensible claim
A **heterogeneous residual-specialist mixer** can reduce the damage of forcing one inductive bias everywhere, while preserving a strong simple base.

## Weakest claim you should NOT make
Do **not** claim state-of-the-art across all 9 datasets unless the full run really proves it.

## Kill test
If `hersmix` does **not** beat the strongest internal baselines (`base_only`, `base+wavelet_only`) on most pilot datasets, reject the idea.

In [ ]:
# ============================================
# CONFIG
# ============================================

RUN_MODE = "pilot_then_full"   # options: "pilot_only", "pilot_then_full", "full_only"
AUTO_FULL_IF_PILOT_GOOD = True

PILOT_DATASETS = ["etth1", "weather", "traffic", "ili"]

# pilot success rule:
# 1) hersmix beats base_only on >= 3 pilot datasets by RMSE
# 2) hersmix beats wavelet_only on >= 3 pilot datasets by RMSE
PILOT_THRESH_BASE = 3
PILOT_THRESH_WAVE = 3

SEED = 42
ROOT_DIR = "/data/Sajjan_Singh/spml/wavelet_seq_project"

In [ ]:
# ============================================
# IMPORTS AND SETUP
# ============================================

from pathlib import Path
import sys
import importlib
import gc
import json
import random
import copy
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path(ROOT_DIR).resolve()
SCRIPTS_DIR = ROOT / "scripts"
RUNNER_PATH = SCRIPTS_DIR / "phase5_adaptive_wavelet_model.py"

if not RUNNER_PATH.exists():
    raise FileNotFoundError(f"Missing runner file: {RUNNER_PATH}")

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

import phase5_adaptive_wavelet_model as p5
importlib.reload(p5)

OUT_TABLES = ROOT / "results" / "tables" / "phase11_hersmix"
OUT_PREDS = ROOT / "results" / "predictions" / "phase11_hersmix"
OUT_CKPTS = ROOT / "results" / "checkpoints" / "phase11_hersmix"

OUT_TABLES.mkdir(parents=True, exist_ok=True)
OUT_PREDS.mkdir(parents=True, exist_ok=True)
OUT_CKPTS.mkdir(parents=True, exist_ok=True)

MASTER_PATH = ROOT / "results" / "tables" / "master_all_models_metrics_unified.csv"
BEST_PATH = ROOT / "results" / "tables" / "master_best_by_dataset_unified.csv"

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
else:
    GPU_NAME = "CPU"
    GPU_MEM_GB = 0.0

USE_AMP = bool(torch.cuda.is_available()) and bool(getattr(p5, "TRAIN_CFG", {}).get("use_amp", True))
PATIENCE = int(getattr(p5, "TRAIN_CFG", {}).get("full_patience", 8))

print("Imported p5 from:", RUNNER_PATH)
print("DEVICE:", p5.DEVICE)
print("GPU_NAME:", GPU_NAME)
print("GPU_MEM_GB:", round(GPU_MEM_GB, 2))
print("USE_AMP:", USE_AMP)
print("PATIENCE:", PATIENCE)
print("All datasets:", p5.ALL_DATASETS)

In [ ]:
# ============================================
# FIXED ARCHITECTURE AND TRAINING SETTINGS
# ============================================

ALL_DATASETS = list(p5.ALL_DATASETS)

# keep your best wavelet architecture fixed
WAVE_ARCH = {
    "levels": 2,
    "wavelet_family": "Db4",
    "d_model": 128,
    "num_backbone_blocks": 2,
    "dropout": 0.10,
    "filter_reg_lambda": 1e-4,
    "gate_entropy_lambda": 1e-4,
}

if GPU_MEM_GB >= 80:
    BATCH_MUL = {
        "etth1": 2, "etth2": 2, "ettm1": 2, "ettm2": 2,
        "weather": 2, "electricity": 1, "traffic": 1,
        "exchange": 2, "ili": 2,
    }
else:
    BATCH_MUL = {ds: 1 for ds in ALL_DATASETS}

MIXER_CFG = {
    "lambda_resid_aux": 0.15,
    "lambda_router_sparse": 2e-4,
    "lambda_router_entropy": 2e-4,
    "lambda_router_proxy": 1e-3,
    "lambda_selective": 1.0,
    "selector_keep_ratio": 0.70,  # keep easiest 70% samples per batch
    "router_hidden": 48,
    "fourier_hidden": 64,
    "local_hidden": 64,
    "local_kernel": 5,
}

print("WAVE_ARCH:", WAVE_ARCH)
print("MIXER_CFG:", MIXER_CFG)
print("BATCH_MUL:", BATCH_MUL)

In [ ]:
# ============================================
# HELPERS
# ============================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_true - y_pred) ** 2))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(mse(y_true, y_pred)))

def resolve_seq_len(ds):
    seq_candidates = p5.resolve_seq_candidates(ds, "searched")
    return int(seq_candidates[min(1, len(seq_candidates) - 1)])

def resolve_batch_size(ds):
    return int(p5.DEFAULT_BATCH_SIZE[ds] * BATCH_MUL[ds])

def resolve_epochs(ds):
    return int(p5.DEFAULT_EPOCHS[ds])

def metric_row_from_npz(dataset, model, family, seq_len, pred_len, pred_file_rel):
    p = ROOT / str(pred_file_rel)
    arr = np.load(p, allow_pickle=True)
    preds = np.asarray(arr["preds"], dtype=np.float64).reshape(-1)
    trues = np.asarray(arr["trues"], dtype=np.float64).reshape(-1)
    return {
        "dataset": dataset,
        "model": model,
        "family": family,
        "seq_len": int(seq_len),
        "pred_len": int(pred_len),
        "test_mse": mse(trues, preds),
        "test_mae": mae(trues, preds),
        "test_rmse": rmse(trues, preds),
        "prediction_file": str(pred_file_rel),
    }

def moving_avg_1d(x, kernel):
    if kernel <= 1:
        return x
    pad = kernel // 2
    x3 = x.unsqueeze(1)
    xpad = F.pad(x3, (pad, pad), mode="replicate")
    out = F.avg_pool1d(xpad, kernel_size=kernel, stride=1)
    return out.squeeze(1)

def select_easy_mask(score, keep_ratio=0.7):
    # lower score = easier / more reliable
    B = score.shape[0]
    k = max(1, int(round(B * keep_ratio)))
    thresh = torch.kthvalue(score.detach(), k).values
    mask = (score <= thresh).float()
    return mask

In [ ]:
# ============================================
# MODEL COMPONENTS
# ============================================

class DLinearTargetOnly(nn.Module):
    def __init__(self, seq_len, pred_len, kernel_size=25):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.kernel_size = int(kernel_size)
        self.trend_linear = nn.Linear(self.seq_len, self.pred_len)
        self.seasonal_linear = nn.Linear(self.seq_len, self.pred_len)

    def forward(self, x_target):
        trend = moving_avg_1d(x_target, self.kernel_size)
        seasonal = x_target - trend
        return self.trend_linear(trend) + self.seasonal_linear(seasonal)

def complexity_proxy(x_target):
    # x_target: [B,T]
    spec = torch.fft.rfft(x_target, dim=1)
    power = spec.real.pow(2) + spec.imag.pow(2)
    total_power = power.sum(dim=1) + 1e-8
    Fbins = power.shape[1]
    split = max(1, int(round(Fbins * 0.50)))
    high_power = power[:, split:].sum(dim=1)
    high_ratio = high_power / total_power

    tv = (x_target[:, 1:] - x_target[:, :-1]).abs().mean(dim=1)
    scale = x_target.abs().mean(dim=1) + 1e-6
    tv_norm = torch.tanh(tv / scale)

    x1 = x_target[:, 1:]
    x0 = x_target[:, :-1]
    num = (x1 * x0).mean(dim=1)
    den = x1.pow(2).mean(dim=1).sqrt() * x0.pow(2).mean(dim=1).sqrt() + 1e-6
    lag1_corr = num / den
    decor = 1.0 - lag1_corr.abs()

    proxy = 0.50 * high_ratio + 0.30 * tv_norm + 0.20 * decor
    return proxy.clamp(0.0, 1.0)

class PeriodicResidualExpert(nn.Module):
    '''
    Frequency-aware residual corrector on target channel only.
    '''
    def __init__(self, seq_len, pred_len, hidden=64):
        super().__init__()
        self.seq_len = int(seq_len)
        self.pred_len = int(pred_len)
        self.net = nn.Sequential(
            nn.Linear(self.seq_len * 2, hidden),
            nn.GELU(),
            nn.Linear(hidden, self.pred_len)
        )

    def forward(self, x_target):
        fft = torch.fft.rfft(x_target, dim=1)
        real = fft.real
        imag = fft.imag
        feat = torch.cat([real, imag], dim=1)
        # pad/truncate to 2*seq_len for fixed linear input
        target_dim = self.seq_len * 2
        if feat.shape[1] < target_dim:
            feat = F.pad(feat, (0, target_dim - feat.shape[1]))
        elif feat.shape[1] > target_dim:
            feat = feat[:, :target_dim]
        return self.net(feat)

class LocalResidualExpert(nn.Module):
    '''
    Short-range local-pattern residual corrector.
    '''
    def __init__(self, seq_len, pred_len, hidden=64, kernel=5):
        super().__init__()
        self.conv1 = nn.Conv1d(1, hidden, kernel_size=kernel, padding=kernel//2)
        self.conv2 = nn.Conv1d(hidden, hidden, kernel_size=kernel, padding=kernel//2)
        self.proj = nn.Linear(seq_len * hidden, pred_len)

    def forward(self, x_target):
        x = x_target.unsqueeze(1)      # [B,1,T]
        x = F.gelu(self.conv1(x))
        x = F.gelu(self.conv2(x))
        x = x.flatten(1)
        return self.proj(x)

class WaveletResidualExpert(nn.Module):
    '''
    Full multivariate wavelet residual expert using your current module.
    '''
    def __init__(self, input_dim, pred_len):
        super().__init__()
        self.model = p5.AdaptiveWaveletMixer(
            input_dim=int(input_dim),
            pred_len=int(pred_len),
            d_model=int(WAVE_ARCH["d_model"]),
            levels=int(WAVE_ARCH["levels"]),
            wavelet_family=WAVE_ARCH["wavelet_family"],
            num_backbone_blocks=int(WAVE_ARCH["num_backbone_blocks"]),
            dropout=float(WAVE_ARCH["dropout"]),
            filter_reg_lambda=float(WAVE_ARCH["filter_reg_lambda"]),
            gate_entropy_lambda=float(WAVE_ARCH["gate_entropy_lambda"]),
        )

    def forward(self, x_scaled):
        pred_scaled, aux = self.model(x_scaled)
        return pred_scaled

class Router(nn.Module):
    '''
    Outputs mixture over residual experts:
    [none, periodic, local, wavelet]
    '''
    def __init__(self, hidden=48):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, hidden),
            nn.GELU(),
            nn.Linear(hidden, 4)
        )

    def forward(self, x_target):
        proxy = complexity_proxy(x_target)
        mean_abs = x_target.abs().mean(dim=1)
        std = x_target.std(dim=1)
        if x_target.shape[1] >= 2:
            last_diff = (x_target[:, -1] - x_target[:, -2]).abs()
        else:
            last_diff = torch.zeros_like(mean_abs)
        trend_strength = moving_avg_1d(x_target, 25).std(dim=1) / (std + 1e-6)

        feats = torch.stack([
            proxy,
            torch.tanh(std / (mean_abs + 1e-6)),
            torch.tanh(last_diff / (mean_abs + 1e-6)),
            torch.tanh(trend_strength),
            torch.ones_like(proxy),
        ], dim=1)
        logits = self.net(feats)
        w = torch.softmax(logits, dim=1)
        return w, proxy

class HeRSMix(nn.Module):
    '''
    Modes:
    - base_only
    - wavelet_only
    - hersmix
    '''
    def __init__(self, seq_len, pred_len, input_dim, target_idx, mode="hersmix"):
        super().__init__()
        assert mode in ["base_only", "wavelet_only", "hersmix"]
        self.mode = mode
        self.target_idx = int(target_idx)

        self.base = DLinearTargetOnly(seq_len=seq_len, pred_len=pred_len, kernel_size=25)
        self.periodic = PeriodicResidualExpert(seq_len=seq_len, pred_len=pred_len, hidden=MIXER_CFG["fourier_hidden"])
        self.local = LocalResidualExpert(seq_len=seq_len, pred_len=pred_len, hidden=MIXER_CFG["local_hidden"], kernel=MIXER_CFG["local_kernel"])
        self.wavelet = WaveletResidualExpert(input_dim=input_dim, pred_len=pred_len)
        self.router = Router(hidden=MIXER_CFG["router_hidden"])

    def forward(self, x_scaled):
        x_target = x_scaled[:, :, self.target_idx]  # [B,T]
        base_pred = self.base(x_target)

        if self.mode == "base_only":
            B = x_scaled.shape[0]
            zeros = torch.zeros_like(base_pred)
            aux = {
                "weights": torch.zeros(B, 4, device=x_scaled.device),
                "proxy": complexity_proxy(x_target),
                "base_pred": base_pred,
                "p_pred": zeros,
                "l_pred": zeros,
                "w_pred": zeros,
                "final_pred": base_pred,
            }
            return base_pred, aux

        p_pred = self.periodic(x_target)
        l_pred = self.local(x_target)
        w_pred = self.wavelet(x_scaled)

        if self.mode == "wavelet_only":
            final_pred = base_pred + w_pred
            B = x_scaled.shape[0]
            weights = torch.zeros(B, 4, device=x_scaled.device)
            weights[:, 3] = 1.0
            aux = {
                "weights": weights,
                "proxy": complexity_proxy(x_target),
                "base_pred": base_pred,
                "p_pred": p_pred,
                "l_pred": l_pred,
                "w_pred": w_pred,
                "final_pred": final_pred,
            }
            return final_pred, aux

        weights, proxy = self.router(x_target)
        # weights columns: [none, periodic, local, wavelet]
        none_w = weights[:, 0:1]
        p_w = weights[:, 1:2]
        l_w = weights[:, 2:3]
        w_w = weights[:, 3:4]

        resid = p_w * p_pred + l_w * l_pred + w_w * w_pred
        final_pred = base_pred + resid

        aux = {
            "weights": weights,
            "proxy": proxy,
            "base_pred": base_pred,
            "p_pred": p_pred,
            "l_pred": l_pred,
            "w_pred": w_pred,
            "final_pred": final_pred,
        }
        return final_pred, aux

def hersmix_loss(pred_scaled, y_scaled, aux, mode):
    B = y_scaled.shape[0]
    per_sample_mse = ((pred_scaled - y_scaled) ** 2).mean(dim=1)

    if mode == "base_only":
        base = per_sample_mse.mean()
        return base, {
            "final_mse": float(base.detach().cpu().item()),
            "selective_keep": 1.0,
            "router_sparse": 0.0,
            "router_entropy": 0.0,
            "router_proxy": 0.0,
            "resid_aux": 0.0,
        }

    base_pred = aux["base_pred"]
    p_pred = aux["p_pred"]
    l_pred = aux["l_pred"]
    w_pred = aux["w_pred"]
    weights = aux["weights"]
    proxy = aux["proxy"]

    # residual specialization
    resid_target = (y_scaled - base_pred).detach()
    resid_aux = (
        F.mse_loss(p_pred, resid_target)
        + F.mse_loss(l_pred, resid_target)
        + F.mse_loss(w_pred, resid_target)
    ) / 3.0

    if mode == "wavelet_only":
        base = per_sample_mse.mean()
        total = base + MIXER_CFG["lambda_resid_aux"] * resid_aux
        return total, {
            "final_mse": float(base.detach().cpu().item()),
            "selective_keep": 1.0,
            "router_sparse": 0.0,
            "router_entropy": 0.0,
            "router_proxy": 0.0,
            "resid_aux": float(resid_aux.detach().cpu().item()),
        }

    # selective weighting by disagreement among residual experts
    disagreement = torch.stack([
        (p_pred - l_pred).pow(2).mean(dim=1),
        (p_pred - w_pred).pow(2).mean(dim=1),
        (l_pred - w_pred).pow(2).mean(dim=1),
    ], dim=1).mean(dim=1)

    easy_mask = select_easy_mask(disagreement, keep_ratio=MIXER_CFG["selector_keep_ratio"])
    weighted_mse = (per_sample_mse * easy_mask).sum() / (easy_mask.sum() + 1e-6)

    # router regularization
    none_w = weights[:, 0]
    p_w = weights[:, 1]
    l_w = weights[:, 2]
    w_w = weights[:, 3]

    active = 1.0 - none_w
    router_sparse = active.mean()
    router_entropy = -(weights.clamp_min(1e-8).log() * weights).sum(dim=1).mean()

    # align wavelet usage with complexity proxy, but softly
    router_proxy = F.mse_loss(w_w, proxy)

    total = (
        MIXER_CFG["lambda_selective"] * weighted_mse
        + MIXER_CFG["lambda_resid_aux"] * resid_aux
        + MIXER_CFG["lambda_router_sparse"] * router_sparse
        + MIXER_CFG["lambda_router_entropy"] * router_entropy
        + MIXER_CFG["lambda_router_proxy"] * router_proxy
    )

    return total, {
        "final_mse": float(weighted_mse.detach().cpu().item()),
        "selective_keep": float(easy_mask.mean().detach().cpu().item()),
        "router_sparse": float(router_sparse.detach().cpu().item()),
        "router_entropy": float(router_entropy.detach().cpu().item()),
        "router_proxy": float(router_proxy.detach().cpu().item()),
        "resid_aux": float(resid_aux.detach().cpu().item()),
    }

In [ ]:
# ============================================
# TRAIN / EVAL
# ============================================

@torch.no_grad()
def evaluate_model(model, loader, target_mean, target_std):
    model.eval()
    preds = []
    trues = []
    weight_rows = []
    proxy_rows = []

    for batch in loader:
        x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
        y_raw = batch["y_raw"].cpu().numpy()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            pred_scaled, aux = model(x)

        pred_raw = pred_scaled.detach().cpu().numpy() * target_std + target_mean
        preds.append(pred_raw)
        trues.append(y_raw)
        weight_rows.append(aux["weights"].detach().cpu().numpy())
        proxy_rows.append(aux["proxy"].detach().cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)
    weights = np.concatenate(weight_rows, axis=0)
    proxy = np.concatenate(proxy_rows, axis=0)

    avg_weights = weights.mean(axis=0)

    return {
        "preds": preds,
        "trues": trues,
        "mse": mse(trues.reshape(-1), preds.reshape(-1)),
        "mae": mae(trues.reshape(-1), preds.reshape(-1)),
        "rmse": rmse(trues.reshape(-1), preds.reshape(-1)),
        "avg_none_w": float(avg_weights[0]),
        "avg_periodic_w": float(avg_weights[1]),
        "avg_local_w": float(avg_weights[2]),
        "avg_wavelet_w": float(avg_weights[3]),
        "avg_proxy": float(proxy.mean()),
    }

def train_one_run(dataset_name, mode):
    set_seed(SEED)

    bundle = p5.load_bundle(dataset_name, input_mode="multivariate")
    pred_len = p5.resolve_pred_len(dataset_name, "long")
    seq_len = resolve_seq_len(dataset_name)
    batch_size = resolve_batch_size(dataset_name)
    epochs = resolve_epochs(dataset_name)

    train_loader, val_loader, test_loader = p5.make_loaders(bundle, seq_len, pred_len, batch_size)

    target_idx = bundle["target_idx"]
    target_mean = float(bundle["scaler_mean"][target_idx])
    target_std = float(bundle["scaler_std"][target_idx])

    model = HeRSMix(
        seq_len=seq_len,
        pred_len=pred_len,
        input_dim=int(bundle["values_scaled"].shape[1]),
        target_idx=int(target_idx),
        mode=mode,
    ).to(p5.DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5,
        foreach=False,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_val_rmse = float("inf")
    best_epoch = -1
    best_state = None
    patience_left = PATIENCE

    history = []
    t0 = time.time()

    for epoch in range(1, epochs + 1):
        model.train()

        loss_vals = []
        final_mse_vals = []
        selective_vals = []
        sparse_vals = []
        entropy_vals = []
        proxy_vals = []
        resid_aux_vals = []

        for batch in train_loader:
            x = batch["x_scaled"].to(p5.DEVICE, non_blocking=True)
            y = batch["y_scaled"].to(p5.DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                pred_scaled, aux = model(x)
                loss, parts = hersmix_loss(pred_scaled, y, aux, mode)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            loss_vals.append(float(loss.detach().cpu().item()))
            final_mse_vals.append(parts["final_mse"])
            selective_vals.append(parts["selective_keep"])
            sparse_vals.append(parts["router_sparse"])
            entropy_vals.append(parts["router_entropy"])
            proxy_vals.append(parts["router_proxy"])
            resid_aux_vals.append(parts["resid_aux"])

        val_eval = evaluate_model(model, val_loader, target_mean, target_std)
        val_rmse = float(val_eval["rmse"])

        history.append({
            "dataset": dataset_name,
            "mode": mode,
            "epoch": epoch,
            "train_loss": float(np.mean(loss_vals)),
            "train_final_mse": float(np.mean(final_mse_vals)),
            "train_selective_keep": float(np.mean(selective_vals)),
            "train_router_sparse": float(np.mean(sparse_vals)),
            "train_router_entropy": float(np.mean(entropy_vals)),
            "train_router_proxy": float(np.mean(proxy_vals)),
            "train_resid_aux": float(np.mean(resid_aux_vals)),
            "val_rmse": val_rmse,
            "val_mae": float(val_eval["mae"]),
            "val_mse": float(val_eval["mse"]),
            "val_none_w": float(val_eval["avg_none_w"]),
            "val_periodic_w": float(val_eval["avg_periodic_w"]),
            "val_local_w": float(val_eval["avg_local_w"]),
            "val_wavelet_w": float(val_eval["avg_wavelet_w"]),
            "val_proxy": float(val_eval["avg_proxy"]),
        })

        print(
            f"[{dataset_name} | {mode}] epoch {epoch:03d} | "
            f"train_loss={np.mean(loss_vals):.6f} | val_rmse={val_rmse:.6f} | "
            f"weights=(none={val_eval['avg_none_w']:.3f}, p={val_eval['avg_periodic_w']:.3f}, "
            f"l={val_eval['avg_local_w']:.3f}, w={val_eval['avg_wavelet_w']:.3f}) | "
            f"proxy={val_eval['avg_proxy']:.3f} | batch={batch_size}"
        )

        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_left = PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{dataset_name} | {mode}] early stopping at epoch {epoch}")
                break

    model.load_state_dict(best_state)
    test_eval = evaluate_model(model, test_loader, target_mean, target_std)

    ckpt_path = OUT_CKPTS / f"{dataset_name}_{mode}.pt"
    pred_path = OUT_PREDS / f"{dataset_name}_{mode}_test_predictions.npz"

    torch.save({
        "dataset": dataset_name,
        "mode": mode,
        "wave_arch": WAVE_ARCH,
        "mixer_cfg": MIXER_CFG,
        "seq_len": seq_len,
        "pred_len": pred_len,
        "best_epoch": best_epoch,
        "state_dict": model.state_dict(),
    }, ckpt_path)

    np.savez_compressed(
        pred_path,
        preds=test_eval["preds"],
        trues=test_eval["trues"],
        seq_len=seq_len,
        pred_len=pred_len,
        avg_none_w=np.array([test_eval["avg_none_w"]], dtype=np.float32),
        avg_periodic_w=np.array([test_eval["avg_periodic_w"]], dtype=np.float32),
        avg_local_w=np.array([test_eval["avg_local_w"]], dtype=np.float32),
        avg_wavelet_w=np.array([test_eval["avg_wavelet_w"]], dtype=np.float32),
        avg_proxy=np.array([test_eval["avg_proxy"]], dtype=np.float32),
    )

    row = {
        "dataset": dataset_name,
        "mode": mode,
        "model": f"HeRSMix_{mode}",
        "family": "wavelet_regime_adaptive",
        "seq_len": seq_len,
        "pred_len": pred_len,
        "batch_size": batch_size,
        "epochs_target": epochs,
        "best_epoch": int(best_epoch),
        "train_seconds": float(time.time() - t0),
        "checkpoint": str(ckpt_path.relative_to(ROOT)),
        "prediction_file": str(pred_path.relative_to(ROOT)),
        "test_mse": float(test_eval["mse"]),
        "test_mae": float(test_eval["mae"]),
        "test_rmse": float(test_eval["rmse"]),
        "avg_none_w": float(test_eval["avg_none_w"]),
        "avg_periodic_w": float(test_eval["avg_periodic_w"]),
        "avg_local_w": float(test_eval["avg_local_w"]),
        "avg_wavelet_w": float(test_eval["avg_wavelet_w"]),
        "avg_proxy": float(test_eval["avg_proxy"]),
    }

    hist_df = pd.DataFrame(history)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row, hist_df

In [ ]:
# ============================================
# PILOT RUN
# ============================================

pilot_rows = []
pilot_hist = []

if RUN_MODE in ["pilot_only", "pilot_then_full"]:
    print("Starting pilot...")
    for ds in PILOT_DATASETS:
        for mode in ["base_only", "wavelet_only", "hersmix"]:
            print("\n" + "=" * 120)
            print(f"PILOT | dataset={ds} | mode={mode}")
            print("=" * 120)
            row, hist_df = train_one_run(ds, mode)
            pilot_rows.append(row)
            pilot_hist.append(hist_df)

    pilot_df = pd.DataFrame(pilot_rows)
    pilot_hist_df = pd.concat(pilot_hist, ignore_index=True)

    pilot_metrics_csv = OUT_TABLES / "pilot_metrics.csv"
    pilot_history_csv = OUT_TABLES / "pilot_history.csv"
    pilot_df.to_csv(pilot_metrics_csv, index=False)
    pilot_hist_df.to_csv(pilot_history_csv, index=False)

    print("\nSaved:", pilot_metrics_csv)
    print("Saved:", pilot_history_csv)
    display(pilot_df)

    pivot = pilot_df.pivot(index="dataset", columns="mode", values="test_rmse").reset_index()
    pivot["hersmix_vs_base_gain"] = pivot["base_only"] - pivot["hersmix"]
    pivot["hersmix_vs_wave_gain"] = pivot["wavelet_only"] - pivot["hersmix"]

    base_wins = int((pivot["hersmix_vs_base_gain"] > 0).sum())
    wave_wins = int((pivot["hersmix_vs_wave_gain"] > 0).sum())

    pilot_summary = pivot.copy()
    pilot_summary["pilot_pass_base"] = base_wins
    pilot_summary["pilot_pass_wave"] = wave_wins

    pilot_summary_csv = OUT_TABLES / "pilot_summary.csv"
    pilot_summary.to_csv(pilot_summary_csv, index=False)

    print("\nSaved:", pilot_summary_csv)
    display(pilot_summary)

    PILOT_GOOD = (base_wins >= PILOT_THRESH_BASE) and (wave_wins >= PILOT_THRESH_WAVE)
    print("\nPILOT DECISION")
    print(f"hersmix beats base_only on {base_wins}/{len(PILOT_DATASETS)} pilot datasets")
    print(f"hersmix beats wavelet_only on {wave_wins}/{len(PILOT_DATASETS)} pilot datasets")
    print("PILOT_GOOD =", PILOT_GOOD)
else:
    PILOT_GOOD = False

In [ ]:
# ============================================
# FULL RUN (hersmix only) IF PILOT IS GOOD
# ============================================

full_rows = []
full_hist = []

RUN_FULL = (
    (RUN_MODE == "full_only") or
    (RUN_MODE == "pilot_then_full" and AUTO_FULL_IF_PILOT_GOOD and PILOT_GOOD)
)

print("RUN_FULL =", RUN_FULL)

if RUN_FULL:
    for ds in ALL_DATASETS:
        print("\n" + "=" * 120)
        print(f"FULL RUN | dataset={ds} | mode=hersmix")
        print("=" * 120)
        row, hist_df = train_one_run(ds, "hersmix")
        full_rows.append(row)
        full_hist.append(hist_df)

    full_df = pd.DataFrame(full_rows)
    full_hist_df = pd.concat(full_hist, ignore_index=True)

    full_metrics_csv = OUT_TABLES / "full_metrics.csv"
    full_history_csv = OUT_TABLES / "full_history.csv"
    full_df.to_csv(full_metrics_csv, index=False)
    full_hist_df.to_csv(full_history_csv, index=False)

    print("\nSaved:", full_metrics_csv)
    print("Saved:", full_history_csv)
    display(full_df)
else:
    print("Full run skipped. Stop here if pilot failed.")

In [ ]:
# ============================================
# MERGE WITH CURRENT MASTER AND COMPARE AGAINST CURRENT BEST
# ============================================

if RUN_FULL:
    if not MASTER_PATH.exists():
        raise FileNotFoundError(f"Missing master file: {MASTER_PATH}")
    if not BEST_PATH.exists():
        raise FileNotFoundError(f"Missing best file: {BEST_PATH}")

    master = pd.read_csv(MASTER_PATH)
    best_old = pd.read_csv(BEST_PATH)

    master_candidate = master[master["model"] != "HeRSMix_hersmix"].copy()

    new_rows = []
    for _, r in full_df.iterrows():
        new_rows.append(
            metric_row_from_npz(
                dataset=r["dataset"],
                model=r["model"],
                family=r["family"],
                seq_len=r["seq_len"],
                pred_len=r["pred_len"],
                pred_file_rel=r["prediction_file"],
            )
        )

    new_df = pd.DataFrame(new_rows)
    master_candidate = pd.concat([master_candidate, new_df], ignore_index=True)
    master_candidate = master_candidate.sort_values(["dataset", "family", "model"]).reset_index(drop=True)

    best_candidate = (
        master_candidate.sort_values(["dataset", "test_rmse", "test_mae", "test_mse"])
                        .groupby("dataset", as_index=False)
                        .first()[["dataset", "model", "family", "test_mse", "test_mae", "test_rmse"]]
    )

    candidate_master_csv = ROOT / "results" / "tables" / "master_all_models_metrics_unified_hersmix_candidate.csv"
    candidate_best_csv = ROOT / "results" / "tables" / "master_best_by_dataset_unified_hersmix_candidate.csv"

    master_candidate.to_csv(candidate_master_csv, index=False)
    best_candidate.to_csv(candidate_best_csv, index=False)

    compare = best_old.rename(columns={
        "model": "old_best_model",
        "family": "old_best_family",
        "test_mse": "old_best_mse",
        "test_mae": "old_best_mae",
        "test_rmse": "old_best_rmse",
    }).merge(
        full_df[[
            "dataset", "model", "test_mse", "test_mae", "test_rmse",
            "avg_none_w", "avg_periodic_w", "avg_local_w", "avg_wavelet_w", "avg_proxy"
        ]].rename(columns={
            "model": "hersmix_model",
            "test_mse": "hersmix_mse",
            "test_mae": "hersmix_mae",
            "test_rmse": "hersmix_rmse",
        }),
        on="dataset",
        how="left"
    )

    compare["rmse_gain_abs"] = compare["old_best_rmse"] - compare["hersmix_rmse"]
    compare["rmse_gain_pct"] = 100.0 * compare["rmse_gain_abs"] / compare["old_best_rmse"]

    compare["mae_gain_abs"] = compare["old_best_mae"] - compare["hersmix_mae"]
    compare["mae_gain_pct"] = 100.0 * compare["mae_gain_abs"] / compare["old_best_mae"]

    compare_csv = OUT_TABLES / "full_vs_current_best.csv"
    compare.to_csv(compare_csv, index=False)

    wins = int((compare["rmse_gain_abs"] > 0).sum())
    losses = int((compare["rmse_gain_abs"] < 0).sum())
    ties = int((compare["rmse_gain_abs"] == 0).sum())

    print("\nSaved:", candidate_master_csv)
    print("Saved:", candidate_best_csv)
    print("Saved:", compare_csv)
    display(compare)
    display(best_candidate)

    print("\nFINAL DECISION")
    print(f"HeRSMix wins on {wins}/{len(ALL_DATASETS)} datasets by RMSE")
    print(f"Loses on {losses}/{len(ALL_DATASETS)} datasets")
    print(f"Ties on {ties}/{len(ALL_DATASETS)} datasets")

    if wins >= 4:
        print("RECOMMENDATION: promising enough to keep and analyze further.")
    else:
        print("RECOMMENDATION: not strong enough overall; reject or redesign.")
else:
    print("No full-run merge because RUN_FULL is False.")

## Expected metrics table layout

Pilot and full metrics tables contain:

- `dataset`
- `mode`
- `model`
- `family`
- `seq_len`
- `pred_len`
- `batch_size`
- `best_epoch`
- `test_mse`
- `test_mae`
- `test_rmse`
- `avg_none_w`
- `avg_periodic_w`
- `avg_local_w`
- `avg_wavelet_w`
- `avg_proxy`

## Failure signals to monitor

Reject the idea if any of these happen:

1. `hersmix` fails to beat `base_only` and `wavelet_only` on most pilot datasets
2. router collapses:
   - `avg_none_w` ~ 1 everywhere, or
   - `avg_wavelet_w` / another expert dominates everywhere
3. selective keep ratio is too low and model stops learning useful corrections
4. full run improves no datasets against current best